# Let's build GPT: from scratch, in code, spelled out.

Following Andrej Karpathy's tutorial: [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=7)

- Companion Repo: [karpathy/ng-video-lecture](https://github.com/karpathy/ng-video-lecture)
- Attention is all you need paper: https://arxiv.org/abs/1706.03762
- Dataset: Tiny Shakespeare (`input.txt`)

In [ ]:
# This will have input characters, tokenization (or encoding, decoding), batch_size, and block_size discussion.

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

### 1. Read and Inspect Dataset

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")
print("First 250 characters:\n", text[:250])

Length of dataset in characters: 1115394
First 250 characters:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



### 2. Vocabulary and Character-Level Tokenizer

In [6]:
# All unique characters occurring in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocab size: {vocab_size}")

# Create mapping from characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]          # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [14]:
# Tokensizers can be anything. Like Google uses https://github.com/google/sentencepiece, sub-word.
# OpenAI uses BPE (Byte Pair Encoding)

import tiktoken
enc = tiktoken.get_encoding('gpt2')
print("Similar to our vocabulary with size of 65, tiktoken'izer has ", enc.n_vocab, " vocab size")
print(enc.encode("hii there"))
print(enc.decode([71, 4178, 612]))

Similar to our vocabulary with size of 65, tiktoken'izer has  50257  vocab size
[71, 4178, 612]
hii there


### 3. Encode Dataset and Train / Val Split

In [18]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}, dtype: {data.dtype}")
print(data[:100])

Data shape: torch.Size([1115394]), dtype: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [19]:
# Split: first 90% for training, remaining 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [20]:
# During training, all text is not input in once. 
# Training text is chunked. It is called block_size

block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

[18, 47, 56, 57, 58,  1, 15, 47, 58]

Has multiple examples. Here it has 8 examples.  
18 itself is an example. When no input, 18 is one of the 'label'.  
18, 47 is another example. When 18 is input, 47 is one of the 'label'.  
And so on... until  
18, 47, 56, 57, 58,  1, 15, 47, 58 becomes an example

In [21]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


Multiple chunks at the same time. 
Batch is multiple chunks. Only used for efficiency to keep GPUs busy. It has nothing to do with training itself as different chunks don't talk to each other.
So batch_size can be anything.

In [34]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i + 1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53, 56, 1, 58, 46] the target: 39
when input is [44, 53, 56, 1, 58, 46, 39] the target: 58
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target: 1
when input is [52] the target: 58
when input is [52, 58] the target: 1
when input is [52, 58, 1] the target: 58
when input is [52, 58, 1, 58] the target: 46
when input is [52, 58, 1, 58, 46] the target: 39
when input is [52, 58, 1, 58, 46, 39] the t

In [33]:
# This is the input batch
xb

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])